<a href="https://colab.research.google.com/github/harshith-murali/BIS-Lab/blob/main/Grey_Wolf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


def gwo_feature_selection(X, y, wolves=15, iterations=30,
                          penalty=0.05, seed=42):
    rng = np.random.default_rng(seed)
    features = X.shape[1]

    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev, y_dev, test_size=0.25,
        stratify=y_dev, random_state=seed
    )

    cache = {}

    def evaluate(position):
        # Convert position to a binary feature mask.
        mask = position >= 0.5
        if not mask.any():
            mask[np.argmax(position)] = True

        key = tuple(mask)

        if key not in cache:
            model = DecisionTreeClassifier(
                max_depth=5, random_state=seed
            )
            model.fit(X_train[:, mask], y_train)
            accuracy = accuracy_score(
                y_val, model.predict(X_val[:, mask])
            )
            cost = 1 - accuracy + penalty * mask.sum() / features
            cache[key] = (cost, accuracy)

        cost, accuracy = cache[key]
        return cost, accuracy, mask

    positions = rng.random((wolves, features))
    best_cost = float("inf")
    best_mask = None
    best_accuracy = 0

    for t in range(iterations + 1):
        results = [evaluate(position) for position in positions]
        ranking = np.argsort([result[0] for result in results])

        current = results[ranking[0]]
        if current[0] < best_cost:
            best_cost, best_accuracy = current[:2]
            best_mask = current[2].copy()

        if t == iterations:
            break

        # Three best wolves guide the population.
        leaders = positions[ranking[:3]].copy()
        a = 2 * (1 - t / iterations)
        updated = np.empty_like(positions)

        for i in range(wolves):
            proposals = []

            for leader in leaders:
                r1 = rng.random(features)
                r2 = rng.random(features)

                A = 2 * a * r1 - a
                C = 2 * r2
                D = np.abs(C * leader - positions[i])

                proposals.append(leader - A * D)

            updated[i] = np.mean(proposals, axis=0)

        positions = np.clip(updated, 0, 1)

    # Evaluate the selected subset on untouched test data.
    model = DecisionTreeClassifier(max_depth=5, random_state=seed)
    model.fit(X_dev[:, best_mask], y_dev)
    test_accuracy = accuracy_score(
        y_test, model.predict(X_test[:, best_mask])
    )

    return np.flatnonzero(best_mask), best_accuracy, test_accuracy


# Synthetic demonstration data.
# Replace X with numeric network features and y with intrusion labels.
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    random_state=42
)

selected, validation_accuracy, test_accuracy = (
    gwo_feature_selection(X, y)
)

print("Selected feature indices:", selected.tolist())
print("Number of selected features:", len(selected))
print(f"Validation accuracy: {validation_accuracy:.2%}")
print(f"Test accuracy: {test_accuracy:.2%}")

Selected feature indices: [0, 2, 3, 6, 10, 15, 16]
Number of selected features: 7
Validation accuracy: 80.00%
Test accuracy: 75.00%
